In [13]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, Dense, SimpleRNN
import matplotlib as plt
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupKFold
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import R2Score, RootMeanSquaredError
from tensorflow.keras.preprocessing.sequence import pad_sequences

colab = True

In [116]:
dataset_path = os.getcwd()

if colab:
  dataset_path = os.path.join(dataset_path, os.listdir(dataset_path)[1])
else:
  # print(dataset_path)
  dataset_path = os.path.join(dataset_path, "Final_Dataset")
  # print(os.listdir(dataset_path))
  dataset_path = os.path.join(dataset_path, os.listdir(dataset_path)[0])
# print(dataset_path)

data = pd.read_csv(dataset_path)
# print(data)
X = data.iloc[:, 0:data.shape[1]-1].values
# print(X)
y = data.iloc[:, data.shape[1]-1].values
groups = X[:, 0]
print(groups)
unique_groups = np.unique(groups)
# print(X.shape)
patient_counts = data.groupby('PatientID').size()
patient_counts = np.array(patient_counts.values)
# print(patient_counts)
# Find the maximum number of entries for any patient
max_entries = int(patient_counts.max())
print(X.shape)
scaler = MinMaxScaler()
X_cyclical = X[:, 1:3]
X_sleep = X[:, -1:]

scaled_X = X[:, 3:X.shape[1] - 1]

scaled_X = scaler.fit_transform(scaled_X)

scaled_X = np.concatenate((groups.reshape(-1, 1), X_cyclical, scaled_X, X_sleep), axis = 1)
print(scaled_X.shape)

padding_val = -10

padded_X = None
prev_entry_row = 0

for patient in unique_groups:
  padded_patient_data = []
  current_entry_num = patient_counts[int(patient) - 1]
  padding_required = max_entries - current_entry_num
  # print("Padding Required = ", padding_required)

  if padding_required > 0:
    curr_patient_data = scaled_X[prev_entry_row: prev_entry_row + current_entry_num]
    # print(curr_patient_data, curr_patient_data.shape)
    padding = np.full((padding_required, scaled_X.shape[1]), padding_val)
    padding[:, 0] = int(patient)
    # print(padding)
    padded_X = np.concatenate((padded_X, curr_patient_data, padding), axis = 0)
    prev_entry_row += current_entry_num
    # print(padded_X, padded_X.shape)
  elif padding_required == 0:
    padded_patient_data = scaled_X[prev_entry_row: current_entry_num]
    prev_entry_row += current_entry_num
    padded_X = padded_patient_data

padded_X = padded_X[:, 1:]
print(padded_X.shape)

# print("Final Version = ", padded_X, padded_X.shape)
padded_X = padded_X.reshape(len(unique_groups), max_entries, padded_X.shape[1])

[ 1.  1.  1. ... 25. 25. 25.]
(89678, 10)
(89678, 10)
(102400, 9)


### Create the Model Implementation

Input Layer: __ neurons
Hidden Layers:
Output Layer:

#### Resources:
1. https://www.tensorflow.org/guide/keras/working_with_rnns
2. https://www.tensorflow.org/guide/keras/transfer_learning
3. https://machinelearningmastery.com/understanding-simple-recurrent-neural-networks-in-keras/
4. https://keras.io/api/models/sequential/
5. https://medium.com/@researchgraph/beginners-guide-to-recurrent-neural-networks-rnns-with-keras-7b8eb408caa1

In [117]:
LR = 0.01

In [118]:
def create_rnn_model(input_shape):
    model = Sequential()
    model.add(Masking(mask_value=padding_val, input_shape=input_shape))
    model.add(SimpleRNN(32, activation='tanh', input_shape=input_shape, return_sequences=True))
    model.add(SimpleRNN(16, activation='tanh'))
    model.add(Dense(1))  # Output layer
    model.compile(optimizer=Adam(learning_rate=LR), loss=RootMeanSquaredError(), metrics = [R2Score()])
    return model

In [119]:
##Without Min_Max Scalar
groupKFold = GroupKFold(n_splits=5)

# Groups are the patient IDs (this should align with the 0th column of X)
groups = np.unique(scaled_X[:, 0])

# Flatten the y array for use in training
y_flat = y.flatten()

# Store scores for each split
r2_scores, rmse_vals = [], []

# Apply GroupKFold splits
split_num = 1

for train_index, test_index in groupKFold.split(X=padded_X, groups=groups):
    # Select the training and testing data based on GroupKFold splits
    X_train, X_test = padded_X[train_index], padded_X[test_index]
    y_train, y_test = y_flat[train_index], y_flat[test_index]

    # Create and compile the RNN model
    model = create_rnn_model(input_shape=(X_train.shape[1], X_train.shape[2]))

    # Fit the model to the training data
    history = model.fit(X_train, y_train, epochs=200)

    # Print the training history (optional)
    print(f"Training history for split {split_num}: {history.history}")


    # Predict using the trained model
    y_pred = model.predict(X_test)

    # Calculate RMSE and R2 score
    rmse, r2 = model.evaluate(X_test, y_test)

    # Output results for this split
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {rmse}")
    print(f"First 5 Y_PRED: {y_pred[:5]}")

    # Append the results
    r2_scores.append(r2)
    rmse_vals.append(rmse)

    split_num += 1

# Print the average scores across all splits
print(f"Average R2 Score: {np.mean(r2_scores)}")
print(f"Average RMSE: {np.mean(rmse_vals)}")

Epoch 1/200


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: No gradients provided for any variable.

In [97]:
##Using MinMaxScalar for Standardization
scaler = MinMaxScaler()

##Overall there are 25 groups since there are 25 patient IDs
groupKFold = GroupKFold(n_splits = 5)
min_max_r2_scores, min_max_rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in groupKFold.split(X=padded_X, groups = groups):
    X_train, X_test = padded_X[train_index], padded_X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    print("X_train = ", scaled_X_train, scaled_X_train.shape)
    print("X_test = ", scaled_X_test, scaled_X_test.shape)

    # Create and compile the RNN model
    model = create_rnn_model(input_shape=(X_train.shape[1], X_train.shape[2]))

    # Fit the model to the training data
    history = model.fit(X_train, y_train, epochs=200)

    # Print the training history (optional)
    print(f"Training history for split {split_num}: {history.history}")


    # Predict using the trained model
    y_pred = model.predict(X_test)

    # Calculate RMSE and R2 score
    rmse, r2 = model.evaluate(X_test, y_test)

    # Output results for this split
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {rmse}")
    print(f"First 5 Y_PRED: {y_pred[:5]}")

    # Append the results
    r2_scores.append(r2)
    rmse_vals.append(rmse)

    split_num += 1

# Print the average scores across all splits
print(f"Average R2 Score: {np.mean(r2_scores)}")
print(f"Average RMSE: {np.mean(rmse_vals)}")

ValueError: cannot reshape array of size 180 into shape (20,4096,9)

### Metrics Analysis of Created Model

#### Resources:
1. https://keras.io/api/metrics/regression_metrics/#r2score-class
2. https://keras.io/api/metrics/regression_metrics/#rootmeansquarederror-class

### Create the Visualizations of the Loss Plot

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()